# 02 — Core Architecture and the Mathematics of Automatic Differentiation

> *"Calculus is the most powerful weapon of thought yet devised by the wit of man."* — W. B. Smith

A PINN is a marriage of two engineering miracles: a smooth, parameterized function approximator $u_\theta$, and an algorithm — *reverse-mode automatic differentiation* — that evaluates the exact derivatives of $u_\theta$ with respect to both its inputs $(x,t)$ **and** its parameters $\theta$ at machine precision. The first allows us to *write down* a PDE residual; the second allows us to *minimize* it.

This notebook treats both with the rigor of mathematical physics. We cover:

1. The architecture of $u_\theta$ as a composite, smooth map;
2. Why activation functions matter — analytically and for spectral properties;
3. The chain rule on computational graphs;
4. Numerical vs. symbolic vs. automatic differentiation;
5. Forward-mode AD via dual numbers and Jet algebras;
6. Reverse-mode AD via adjoint operators on the graph;
7. Computing the gradient, Hessian, Laplacian, d'Alembertian, divergence, curl;
8. Higher-order derivatives and the cost-explosion in nested AD;
9. Numerical conditioning, finite-precision pitfalls, and tractable architectures.

## 1. The Neural Network as a Smooth Composite Map

Define the parametric function $u_\theta : \mathbb{R}^{d+1} \to \mathbb{R}^n$ as an $(L+1)$-fold composition

$$
u_\theta \;=\; A^{(L+1)} \,\circ\, \sigma \,\circ\, A^{(L)} \,\circ\, \sigma \,\circ\, \cdots \,\circ\, \sigma \,\circ\, A^{(1)},
$$

where each $A^{(\ell)}(z) = W^{(\ell)} z + b^{(\ell)}$ is an affine map, $\sigma$ is a fixed nonlinear elementwise activation, and $\theta = \{W^{(\ell)},b^{(\ell)}\}_{\ell=1}^{L+1}$.

Let $h^{(0)} = (x,t)\in\mathbb{R}^{d+1}$ and recursively

$$
z^{(\ell)} = W^{(\ell)} h^{(\ell-1)} + b^{(\ell)},\qquad h^{(\ell)} = \sigma(z^{(\ell)}),\qquad \ell=1,\dots,L,
$$

with output $u_\theta(x,t) = W^{(L+1)} h^{(L)} + b^{(L+1)}$. We assume $\sigma \in C^\infty(\mathbb{R})$ (e.g. $\tanh$, $\text{SiLU}(x)=x\sigma_{\text{logistic}}(x)$, $\sin$, $\text{GELU}$). Then by the chain rule for smooth maps,

$$
u_\theta \in C^\infty(\mathbb{R}^{d+1};\mathbb{R}^n) \qquad \forall \theta \in \mathbb{R}^{|\theta|},
$$

and *all derivatives* with respect to $(x,t)$ and $\theta$ exist as elementary functions of the network's intermediate quantities.

### 1.1 Why Width and Depth Both Matter

* The classical UAT (Cybenko) uses arbitrary *width* with depth $L=1$. While theoretically sufficient, single-hidden-layer networks suffer poor sample complexity for compositional functions.
* Depth-separation results (Telgarsky 2016, Eldan–Shamir 2016) prove that some functions expressible by a depth-$L$ network of polynomial width require *exponential* width at depth $L-1$.
* For PINNs, multiple smooth nonlinearities are essential because the PDE residual is itself a *nonlinear* composition of $u_\theta$ with $\mathcal{N}$.

Empirically PINNs use $L \in [4,10]$ and widths $\in [20,256]$. We discuss the spectral-bias-driven choice further in Notebook 3.

## 2. Activation Functions: Smoothness, Saturation, Spectral Content

The activation $\sigma$ has three jobs: provide nonlinearity (for expressivity), provide smoothness (for AD of high-order operators), and provide good gradient signal (for trainability). The PINN community has converged on a handful of choices.

### 2.1 Hyperbolic Tangent

$$\sigma(z) = \tanh z = \frac{e^z - e^{-z}}{e^z + e^{-z}},\quad \sigma'(z) = 1 - \tanh^2 z,\quad \sigma''(z) = -2\tanh z\,(1-\tanh^2 z).$$

Properties: $\sigma\in C^\omega(\mathbb{R})$ (real-analytic), bounded, antisymmetric, derivative $\le 1$.  *Canonical choice* for PINNs; smooth to all orders, gradient flow well-behaved.

### 2.2 Sinusoidal (SIREN, Sitzmann et al. 2020)

$$\sigma(z) = \sin(\omega_0 z).$$

All derivatives bounded; spectrum shifted by the frequency $\omega_0$. Mitigates the *spectral bias* (Notebook 3) by giving the network direct access to high-frequency modes. Initialization scheme: $W \sim \mathcal{U}(-\sqrt{6/n}/\omega_0, +\sqrt{6/n}/\omega_0)$ preserves variance.

### 2.3 SiLU / Swish

$$\sigma(z) = z\,\sigma_{\text{logistic}}(z),\qquad \sigma_{\text{logistic}}(z)=(1+e^{-z})^{-1}.$$

Smooth, monotone on most of $\mathbb{R}$, non-saturating for large positive $z$; favored in modern architectures and PINNs needing both smoothness and ReLU-like behavior.

### 2.4 What to Avoid

ReLU $\sigma(z)=\max(z,0)$ is *not* $C^1$; its weak derivative is the Heaviside step, and second derivative is the Dirac $\delta$. Although the Laplacian $\Delta u_\theta$ exists *a.e.* and AD can be coerced to produce a value via subgradient conventions, the resulting PINN residual is degenerate: it is locally piecewise affine, and second derivatives vanish almost everywhere on the network's linear regions. Thus ReLU PINNs cannot represent functions with non-trivial second derivatives — disqualifying for any second-order PDE. Use only with first-order PDEs and reformulations.

### 2.5 The Smoothness Hierarchy We Need

For a PDE of order $m$, every $\sigma$ must be $C^m(\mathbb{R})$ to ensure $u_\theta \in C^m$ globally. For second-order PDEs (heat, wave, Schrödinger, elasticity), $C^2$ suffices; in practice we use $C^\infty$ activations so the AD machinery is unrestricted.

## 3. The Chain Rule on Computational Graphs

AD is, in one sentence, *the chain rule executed by program transformation on a directed acyclic computational graph*.

### 3.1 Computational Graph as a DAG

Any program computing $y = f(x)$ via elementary operations may be represented as a directed acyclic graph (DAG) $G = (V,E)$ where:

* nodes $v \in V$ are intermediate quantities $v_k$;
* an edge $u \to v$ means $v$ depends directly on $u$ via some elementary local function $\phi$;
* each node carries a **local Jacobian** $\partial v / \partial u$ available in closed form (since the operation is elementary).

**Example.** $y = \tanh(W_2\,\tanh(W_1 x + b_1) + b_2)$ has the graph

$$
x \to v_1 = W_1 x \to v_2 = v_1 + b_1 \to v_3 = \tanh v_2 \to v_4 = W_2 v_3 \to v_5 = v_4 + b_2 \to y = \tanh v_5.
$$

### 3.2 The Multivariate Chain Rule

If $f = f_L \circ \cdots \circ f_1$ with $f_\ell : \mathbb{R}^{n_{\ell-1}} \to \mathbb{R}^{n_\ell}$, then the Jacobian matrix factorizes as a product of local Jacobians:

$$
J_f(x) \;=\; J_{f_L}(z^{(L-1)}) \cdot J_{f_{L-1}}(z^{(L-2)}) \cdots J_{f_1}(x) \;\in\; \mathbb{R}^{n_L \times n_0}.
$$

AD evaluates this matrix product *without ever materializing it as a dense matrix*. Two opposite traversal orders — *forward* (left-multiplying tangent vectors) and *reverse* (right-multiplying cotangent vectors) — give two algorithms with very different complexity profiles.

## 4. Why Not Numerical or Symbolic Differentiation?

Before laying out AD properly, we explain why the two obvious alternatives are inadequate for PINNs.

### 4.1 Numerical (Finite Difference) Differentiation

$$
\frac{\partial u}{\partial x_i}(x) \;\approx\; \frac{u(x+h e_i) - u(x - h e_i)}{2h}.
$$

* **Truncation error** $\mathcal{O}(h^2)$ — biased.
* **Round-off error** $\mathcal{O}(\varepsilon_{\text{mach}}/h)$ — dominates as $h\to 0$. Optimal $h \sim \varepsilon_{\text{mach}}^{1/3} \approx 10^{-5}$ in double precision; even then accuracy is $\sim 10^{-10}$, not $10^{-16}$.
* **Cost**: $\mathcal{O}(d)$ evaluations of $u$ for $\nabla u$. For the Laplacian: $\mathcal{O}(d)$. For Hessian-vector products: $\mathcal{O}(d)$. Catastrophic for high-dim PDEs.
* **Boundary behavior** degenerate near $\partial\Omega$ (one-sided stencils).

Finite differences would defeat the entire point of going mesh-free.

### 4.2 Symbolic Differentiation

Symbolic engines (SymPy, Mathematica) manipulate expression trees, applying $(\sin x)' = \cos x$ recursively. For our 4-layer $\tanh$-MLP, the symbolic gradient of $u_\theta(x,t)$ w.r.t. $x$ is a closed-form expression of size $\mathcal{O}(\text{nodes}\times\text{depth})$. The second derivative roughly squares this. The Laplacian in $d=3$ at depth $L=8$ explodes to gigabytes — **expression swell**.

Worse, the symbolic expression must then be *evaluated* numerically, with no caching of shared sub-expressions, leading to redundant work.

### 4.3 The Promise of AD

Automatic differentiation evaluates derivatives *exactly* (up to floating point), in time proportional to the original function evaluation, and without expression swell, by traversing the computational graph and applying the chain rule operationally. It is the only method satisfying:

1. No truncation error (zero $h$).
2. Round-off at $\varepsilon_{\text{mach}}$ level only.
3. Cost independent of the input dimension *for gradients* (reverse mode).
4. No expression-tree blow-up — intermediates are shared.

## 5. Forward-Mode AD — Dual Numbers and Jets

### 5.1 The Dual Number Algebra

Define the ring $\mathbb{D} = \mathbb{R}[\epsilon]/(\epsilon^2)$ — formal pairs $a + b\epsilon$ with $\epsilon^2 = 0$. Addition and multiplication mirror complex numbers:

$$
(a+b\epsilon)+(c+d\epsilon) = (a+c) + (b+d)\epsilon,\qquad (a+b\epsilon)(c+d\epsilon) = ac + (ad+bc)\epsilon.
$$

For any analytic $f$ on $\mathbb{R}$,

$$
f(a + b\epsilon) \;=\; \sum_{k=0}^\infty \frac{f^{(k)}(a)}{k!}(b\epsilon)^k \;=\; f(a) + f'(a) b\,\epsilon,
$$

since $\epsilon^2=0$. The $\epsilon$-coefficient *is* the derivative — extracted *exactly* and without truncation.

### 5.2 Forward-Mode AD as Dual-Number Evaluation

To compute $J_f(x)\,v$ (a *Jacobian–vector product*, JVP), promote $x \mapsto x + v\epsilon$, push through the network using the dual arithmetic above, and read off the $\epsilon$-part of the output. Cost: roughly $2\times$ the forward pass per direction $v$.

Computing the full Jacobian $J_f \in \mathbb{R}^{m\times n}$ requires $n$ such passes (one per basis vector). For $n\gg m$ — the case for gradients of scalar losses w.r.t. millions of parameters — this is catastrophic. Reverse mode reverses the asymmetry.

### 5.3 Jets and Higher-Order Forward AD

The construction generalizes to $\mathbb{R}[\epsilon]/(\epsilon^{k+1})$ — *truncated Taylor algebras* or *$k$-jets*. Operating in this algebra propagates all Taylor coefficients up to order $k$ in one pass. This is the cleanest way to compute *all directional derivatives along one direction* up to a given order — useful for Taylor-based PINN time integration, but irrelevant for the gradient w.r.t. parameters.

## 6. Reverse-Mode AD — Adjoints and the Cotangent Graph

Reverse mode is the workhorse of deep learning. It computes a *vector–Jacobian product* (VJP) $v^\top J_f(x)$ in time $\mathcal{O}(\text{cost of }f)$ independent of $\dim(x)$.

### 6.1 Adjoint Variables

For any intermediate node $v_k$ with downstream value $y$, define the *adjoint*

$$
\bar v_k \;:=\; \frac{\partial y}{\partial v_k}.
$$

If $v_k$ feeds into nodes $v_{j_1},\dots,v_{j_p}$ via local functions $v_{j_i} = \phi_i(\dots, v_k, \dots)$, the chain rule gives

$$
\bar v_k \;=\; \sum_{i=1}^{p} \bar v_{j_i}\,\frac{\partial \phi_i}{\partial v_k}.
$$

Adjoints are computed by traversing the graph **in reverse topological order**, accumulating contributions.

### 6.2 The Algorithm

Phase 1 (forward sweep): Evaluate $y = f(x)$ from inputs to outputs, *storing intermediate values* $v_k$.

Phase 2 (backward sweep): Initialize $\bar y = 1$ (for scalar $y$); for $k$ from output to input in reverse topological order, accumulate contributions to each predecessor's adjoint by left-multiplying its local Jacobian.

Pseudocode:
```
forward:    v_k = phi_k({v_j : j<k})           for k = 1..N
backward:   v_bar[N] = 1
            for k = N..1:
                for each input j of node k:
                    v_bar[j] += v_bar[k] * d phi_k / d v_j
return:     v_bar[input nodes]
```

### 6.3 Complexity — The Cheap Gradient Principle

**Theorem (Baur–Strassen 1983, Griewank 1989).** For any scalar function $f:\mathbb{R}^n\to\mathbb{R}$ computed by a program of cost $T(f)$, the gradient $\nabla f$ can be computed in cost at most $\omega \cdot T(f)$, with $\omega \le 5$ (typically $2$–$3$).

This is striking: *the gradient of a million-parameter loss is computed in roughly the same time as the loss itself.* This is what makes PINNs feasible.

### 6.4 Memory Cost and Checkpointing

Reverse mode pays in memory: all intermediate $v_k$ along the forward sweep must be retained for the backward pass, costing $\mathcal{O}(T(f))$ space. *Gradient checkpointing* (Chen et al. 2016) trades compute for memory by re-evaluating select segments.

### 6.5 Reverse Mode as Adjoint Calculus

Geometrically, forward mode propagates *tangent vectors* in the input tangent space along $f$; reverse mode propagates *cotangent vectors* (covectors / 1-forms) in the output cotangent space *backwards* along the pullback $f^\ast$. This is the same adjoint formalism used in PDE-constrained optimization, optimal control, and Hamilton–Jacobi theory; deep learning is one instance of a broader principle.

## 7. Computing Differential Operators via AD

We now operationalize PINN derivatives. Let $u_\theta : \mathbb{R}^{d+1}\to\mathbb{R}$ (scalar field). We need:

* Time derivative $\partial_t u_\theta$;
* Spatial gradient $\nabla_x u_\theta = (\partial_{x_1} u_\theta, \dots, \partial_{x_d} u_\theta)$;
* Spatial Hessian $H_x u_\theta = (\partial_{x_i}\partial_{x_j} u_\theta)$;
* Laplacian $\Delta u_\theta = \sum_{i=1}^d \partial_{x_i}^2 u_\theta = \operatorname{tr}(H_x u_\theta)$;
* d'Alembertian $\Box u_\theta = -c^{-2}\partial_t^2 u_\theta + \Delta u_\theta$ (wave equation, signature $(-,+,+,+)$ as physics convention);
* Divergence and curl for vector fields $\mathbf{u}_\theta:\mathbb{R}^{d+1}\to\mathbb{R}^d$.

### 7.1 First Derivatives

Reverse mode gives $\nabla_{(x,t)} u_\theta$ in **one** backward pass: initialize $\bar u = 1$, propagate adjoints back to the input node, and read off the entries corresponding to $x_1,\dots,x_d,t$.

### 7.2 Second Derivatives — The Hessian

There are three strategies, with very different costs.

**(a) Forward-over-reverse.** Compute $g(x) = \nabla u_\theta(x)$ by reverse mode (one VJP), then push forward in dual numbers to obtain Hessian columns one at a time. Cost: $d$ JVPs of a function that itself costs as much as a reverse pass — total $\mathcal{O}(d\,T(u_\theta))$.

**(b) Reverse-over-reverse.** Apply reverse-mode AD to $\nabla u_\theta \cdot v$ for chosen $v$. Conceptually clean but heavy memory.

**(c) Hessian-vector products.** Often we only need $Hv$, not full $H$. This is the *Pearlmutter trick*: differentiate $v\cdot \nabla u_\theta$ once more, cost $\mathcal{O}(T(u_\theta))$.

### 7.3 The Laplacian — Only the Trace is Needed

$\Delta u_\theta = \sum_{i=1}^d \partial_{x_i}^2 u_\theta$. The naive computation evaluates each $\partial_{x_i}^2 u_\theta$ by two AD passes — $\mathcal{O}(d\,T(u_\theta))$.

**Hutchinson's stochastic estimator.** Let $z \in \mathbb{R}^d$ have $\mathbb{E}[z z^\top] = I$. Then

$$
\operatorname{tr}(H) \;=\; \mathbb{E}_z\!\big[z^\top H z\big].
$$

A single Hessian-vector product $Hz$ followed by $z^\top(Hz)$ gives an unbiased estimator. With $K$ samples the variance scales as $1/K$. For large $d$ this is the only tractable Laplacian.

**Stein / Hutchinson++ refinements.** Adamczewski et al., Hu et al. (2024) develop variance-reduced trace estimators tailored to PINN Laplacians.

### 7.4 The d'Alembertian

Combine $\partial_t^2 u_\theta$ (one mixed second derivative via AD on the $t$ coordinate only) and $\Delta u_\theta$:

$$
\Box u_\theta(x,t) \;=\; -\frac{1}{c^2}\partial_t^2 u_\theta(x,t) + \Delta u_\theta(x,t).
$$

Each ingredient is at most $\mathcal{O}(T(u_\theta))$ (with Hutchinson) and the operator scales gracefully to relativistic field PDEs.

### 7.5 Divergence and Curl for Vector Fields

For $\mathbf{u}_\theta = (u_1,\dots,u_d) : \mathbb{R}^{d+1}\to\mathbb{R}^d$:

$$
\nabla\cdot \mathbf{u}_\theta \;=\; \sum_{i=1}^d \partial_{x_i} u_i, \qquad (\nabla\times \mathbf{u}_\theta)_k \;=\; \sum_{ij} \epsilon_{ijk}\,\partial_{x_i} u_j.
$$

Both require partial derivatives that mix output indices and input indices — a *full Jacobian*. Use forward mode if $d$ is small; reverse mode is wasteful here because there is no longer a single scalar output.

### 7.6 Mixed Partials and Schwarz's Theorem

Because $u_\theta \in C^\infty$, mixed partials commute: $\partial_{x_i}\partial_{x_j} u_\theta = \partial_{x_j}\partial_{x_i} u_\theta$. AD respects this only up to floating-point round-off, so in practice we exploit symmetry to halve Hessian computation when needed.

## 8. Differentiating w.r.t. Parameters

The PINN loss $\mathcal{L}(\theta) = \frac{1}{N}\sum_i \big|\mathcal{N}[u_\theta](x_i) - f(x_i)\big|^2$ involves *two* differentiations:

1. **Inner**: $\mathcal{N}[u_\theta](x_i)$ involves derivatives of $u_\theta$ w.r.t. inputs $(x,t)$. Performed by reverse-mode AD on inputs.
2. **Outer**: $\partial \mathcal{L}/\partial \theta$ is the gradient w.r.t. parameters, also via reverse-mode AD.

The trick is that AD must be applied **on a graph that itself contains AD-produced nodes**. Modern frameworks (PyTorch, JAX, TensorFlow) support this seamlessly via *higher-order AD*: each input-AD pass produces a new computational graph whose nodes can themselves be differentiated.

### 8.1 Computational Cost

For a PDE of order $m$ in $d+1$ dimensions, a single PINN training step costs roughly

$$
T_{\text{step}} \;\sim\; \mathcal{O}\!\big(\,c_m \cdot (d+1) \cdot T(u_\theta) \,\big),
$$

where $c_m$ is a polynomial in $m$ (the number of nested AD passes for derivatives up to order $m$) and the factor $(d+1)$ disappears with Hutchinson estimation. Thus PINN cost scales gently with dimension — *the central practical reason for their existence*.

### 8.2 The Cost-Explosion Caveat

Each nested AD layer doubles (approximately) the graph size. A naive 4-th order PDE (e.g. biharmonic $\Delta^2 u = f$) requires 4 nested AD passes, costing $\sim 16\times$ a forward pass. For very high-order PDEs (Cahn–Hilliard, Kuramoto–Sivashinsky), specialized factorizations (Taylor-mode AD, k-jet propagation) are recommended.

## 9. Numerical Conditioning and Precision

### 9.1 Float Precision

Deep learning defaults to float32 ($\varepsilon_{\text{mach}} \approx 10^{-7}$). PINNs typically need float64 ($\varepsilon_{\text{mach}} \approx 10^{-16}$) because:

* Second-derivative AD compounds round-off — the effective error after two AD passes is roughly $\varepsilon_{\text{mach}}^{1/2}$, i.e. $\sim 10^{-3}$ in float32 — incompatible with convergence to high-accuracy PDE solutions.
* The PDE residual $\mathcal{N}[u_\theta] - f$ involves cancellation of large terms (e.g. $u u_x$ vs. $\nu u_{xx}$ in Burgers' equation near shocks); precision matters.

### 9.2 Conditioning of the Loss Landscape

The neural-tangent-kernel matrix $K(\theta) = J_\theta u_\theta\,J_\theta u_\theta^\top$ governs training dynamics. For PINNs, the differential-operator action transforms $K$ into $\mathcal{N}K\mathcal{N}^\top$, which has condition number much larger than $K$ itself — especially for high-order operators. This is one root cause of the *gradient pathology* analyzed in Notebook 3.

### 9.3 Architectural Tricks

* **Sine layers (SIREN, $\omega_0$-init).** Combat spectral bias and explicit-bias-toward-low-frequency.
* **Random Fourier features.** Embed $(x,t) \mapsto [\cos(B(x,t)),\sin(B(x,t))]$ with $B$ Gaussian — Tancik et al. 2020.
* **Adaptive activation slopes.** Jagtap et al. 2020: $\sigma(a^{(\ell)} z)$ with $a^{(\ell)}$ trainable.
* **Hard-constraint output transforms.** Write $u_\theta(x,t) = A(x,t) + B(x,t)\,\tilde u_\theta(x,t)$ where $A$ satisfies the BCs and $B$ vanishes on $\partial\Omega$; this *eliminates* the $\mathcal{L}_b$ term.
* **Domain decomposition** (Notebook 4).

## 10. Worked Symbolic Example: Computing $\partial_t u_\theta + u_\theta\,\partial_x u_\theta - \nu\,\partial_x^2 u_\theta$

For Burgers' equation we will need $r_\theta(x,t) := u_t + u\,u_x - \nu u_{xx}$. Tracing the AD operations:

1. **Forward pass.** Evaluate $u = u_\theta(x,t)$, retaining intermediates $z^{(\ell)}, h^{(\ell)}$.
2. **First reverse pass** w.r.t. inputs. With seeds $\bar u = 1$, backpropagate to obtain $(u_x, u_t) = \nabla_{(x,t)} u_\theta$.
3. **Second reverse pass.** Set up $g = u_x$ as a scalar function of $(x,t)$; backpropagate to obtain $u_{xx} = \partial_x g$.
4. **Form residual.** $r_\theta = u_t + u\cdot u_x - \nu\cdot u_{xx}$.
5. **Loss.** $\mathcal{L}_r = \frac{1}{N_r}\sum_i r_\theta(x^r_i,t^r_i)^2$.
6. **Outer reverse pass** w.r.t. $\theta$. Compute $\partial \mathcal{L}_r/\partial \theta$ — the SGD update direction.

All intermediates and gradients live inside the AD framework's tape, with no symbolic blow-up and no truncation error — exactly the marriage of *function approximation* and *operator embedding* that defines a PINN. Notebook 5 makes this entirely concrete for Burgers.

## 11. Summary and Pointers Forward

We have established:

* The PINN ansatz $u_\theta$ is a smooth composite map whose derivatives of every order exist as elementary functions of intermediates.
* Automatic differentiation computes those derivatives exactly (up to float round-off) in cost proportional to the function — independent of the dimension of the parameter space for scalar gradients.
* Reverse mode is the right tool for parameter gradients; a mix of forward and reverse (and Hutchinson stochastics for traces) is used for input-side derivatives.
* The smoothness of $\sigma$, the choice of architecture, and the use of double precision are all dictated by the algebraic and analytic requirements of evaluating high-order differential operators.

What remains mysterious is *training dynamics*: even with exact gradients, the optimization landscape of a PINN is treacherous. That story — composite losses, collocation strategies, spectral bias, NTK theory, Adam vs. L-BFGS — is the subject of Notebook 3.

## References

1. **Baydin, A. G., Pearlmutter, B. A., Radul, A. A., Siskind, J. M.** *Automatic differentiation in machine learning: a survey.* JMLR 18 (2018).
2. **Griewank, A., Walther, A.** *Evaluating Derivatives: Principles and Techniques of Algorithmic Differentiation.* 2nd ed., SIAM, 2008.
3. **Baur, W., Strassen, V.** *The complexity of partial derivatives.* Theor. Comp. Sci. 22 (1983).
4. **Pearlmutter, B. A.** *Fast exact multiplication by the Hessian.* Neural Computation 6 (1994).
5. **Hutchinson, M. F.** *A stochastic estimator of the trace of the influence matrix...* Comm. Stat. Sim. Comput. 19 (1990).
6. **Sitzmann, V. et al.** *Implicit neural representations with periodic activation functions (SIREN).* NeurIPS 2020.
7. **Tancik, M. et al.** *Fourier features let networks learn high frequency functions in low dimensional domains.* NeurIPS 2020.
8. **Jagtap, A. D., Kawaguchi, K., Karniadakis, G. E.** *Adaptive activation functions accelerate convergence in deep and physics-informed neural networks.* J. Comp. Phys. 404 (2020).